In [16]:
from __future__ import annotations

import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import optuna

import itertools

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts dir:", ARTIFACTS_DIR.resolve())

Artifacts dir: /Users/apple/Desktop/aiee/Abdulmanov_KSBO-13-24_group_1/project/artifacts


In [3]:
transactions = pd.read_csv('../data/transactions_features.csv', parse_dates=['t_dat'])
customers = pd.read_csv('../data/customers_features.csv')
articles = pd.read_csv('../data/articles_features.csv')
customer_cols = [
    'customer_id', 'age', 'age_group', 
    'club_member_status', 'fashion_news_frequency'
]

article_cols = [
    'article_id', 'product_group_name', 'graphical_appearance_name',
    'colour_group_name', 'perceived_colour_value_name', 
    'index_group_name', 'index_name', 'section_name',
    'garment_group_name', 'item_type', 'price_segment'
]

master_exp_02 = transactions.merge(
    customers[customer_cols], 
    on='customer_id', 
    how='left'
)

master_exp_02 = master_exp_02.merge(
    articles[article_cols], 
    on='article_id', 
    how='left'
)

cat_cols = master_exp_02.select_dtypes(include=['object', 'category']).columns
master_exp_02[cat_cols] = master_exp_02[cat_cols].fillna('Unknown')

master_exp_02['age'] = master_exp_02['age'].fillna(master_exp_02['age'].median())

print(master_exp_02.shape)
print(list(master_exp_02.columns))
max_date = master_exp_02['t_dat'].max()
test_start = max_date - pd.Timedelta(days=7)
val_start = test_start - pd.Timedelta(days=7)

train_data = master_exp_02[master_exp_02['t_dat'] < val_start].copy()
val_data = master_exp_02[(master_exp_02['t_dat'] >= val_start) & (master_exp_02['t_dat'] < test_start)].copy()
test_data = master_exp_02[master_exp_02['t_dat'] >= test_start].copy()

/var/folders/yq/ymvjb12x09x0xjvftjt24x2h0000gn/T/ipykernel_62000/4247077022.py:28: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = master_exp_02.select_dtypes(include=['object', 'category']).columns


(1054096, 19)
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id', 'age', 'age_group', 'club_member_status', 'fashion_news_frequency', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'index_group_name', 'index_name', 'section_name', 'garment_group_name', 'item_type', 'price_segment']


In [4]:
customer_cols = ['customer_id', 'age', 'age_group', 'club_member_status']
article_cols = ['article_id', 'product_group_name', 'index_name', 'garment_group_name', 'item_type', 'price_segment']

def create_training_dataset(transactions, all_articles_list, neg_ratio=3):
    positives = transactions[['customer_id', 'article_id']].copy()
    positives['target'] = 1
    
    n_negatives = len(positives) * neg_ratio
    
    negatives = pd.DataFrame({
        'customer_id': np.random.choice(positives['customer_id'], size=n_negatives),
        'article_id': np.random.choice(all_articles_list, size=n_negatives),
        'target': 0
    })

    df = pd.concat([positives, negatives], ignore_index=True)
    
    df = df.drop_duplicates(subset=['customer_id', 'article_id'], keep='first')
    
    pos_count = len(positives)
    df_clean = pd.concat([
        df[df['target'] == 1],
        df[df['target'] == 0].head(pos_count * neg_ratio)
    ])
    
    return df_clean.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

all_article_ids = articles['article_id'].unique()
train_clf = create_training_dataset(train_data, all_article_ids, neg_ratio=3)

train_clf = train_clf.merge(customers[customer_cols], on='customer_id', how='left')
train_clf = train_clf.merge(articles[article_cols], on='article_id', how='left')

train_clf.head()

,customer_id,article_id,target,age,age_group,club_member_status,product_group_name,index_name,garment_group_name,item_type,price_segment
0,60bcea957adb052349c7270598e70dbf56376a00c20639...,858052005,0,33.0,25-34,ACTIVE,Garment Upper body,Ladieswear,Blouses,Fashion,Budget
1,c74e3fa62056d244bd3655522265f983435a8028d6dd65...,535455002,0,21.0,16-24,ACTIVE,Garment Upper body,Ladieswear,Blouses,Fashion,Budget
2,2e32dd7ecfe8fe29fcc643892739dcf52d17b28a8c6b12...,807241026,1,23.0,16-24,ACTIVE,Socks & Tights,Lingeries/Tights,Socks and Tights,Basic,Budget
3,13fb0c2769b91197412108d10ca70dc6a8a867b8d45423...,882900004,1,26.0,25-34,ACTIVE,Garment Lower body,Divided,Trousers Denim,Fashion,Premium
4,c4b9b1916eb2f8cfeba7eeb9fe9f7b7039da3f0612f08c...,862249001,0,50.0,50+,ACTIVE,Accessories,Divided,Accessories,Fashion,Medium


In [5]:
cat_cols = ['age_group', 'club_member_status', 'product_group_name', 
            'index_name', 'garment_group_name', 'item_type', 'price_segment']
features = ['age'] + cat_cols


train_clf[cat_cols] = train_clf[cat_cols].fillna('Unknown')
train_clf['age'] = train_clf['age'].fillna(train_clf['age'].median())

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_clf[col] = le.fit_transform(train_clf[col].astype(str))
    label_encoders[col] = le

X_train = train_clf[features]
y_train = train_clf['target']

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [7]:
val_actual = val_data.groupby('customer_id')['article_id'].apply(list).to_dict()
val_users = list(val_actual.keys())

top_100_candidates = train_data['article_id'].value_counts().head(100).index.tolist()

val_df = pd.DataFrame(
    list(itertools.product(val_users, top_100_candidates)), 
    columns=['customer_id', 'article_id']
)

val_df = val_df.merge(customers[customer_cols], on='customer_id', how='left')
val_df = val_df.merge(articles[article_cols], on='article_id', how='left')

val_df['age'] = val_df['age'].fillna(train_clf['age'].median())
for col in cat_cols:
    val_df[col] = val_df[col].fillna('Unknown')
    known_classes = set(label_encoders[col].classes_)
    val_df[col] = val_df[col].apply(lambda x: x if x in known_classes else 'Unknown')
    val_df[col] = label_encoders[col].transform(val_df[col])

X_val = val_df[features]

val_df['score'] = rf_model.predict_proba(X_val)[:, 1]

top_20_val_df = val_df.sort_values(['customer_id', 'score'], ascending=[True, False]).groupby('customer_id').head(20)
rf_preds_val = top_20_val_df.groupby('customer_id')['article_id'].apply(list).to_dict()

def calculate_metrics(actual_dict, predicted_dict, k=20):
    precisions, recalls = [], []
    for user, actual_items in actual_dict.items():
        actual_set = set(actual_items)
        predicted_set = set(predicted_dict.get(user, [])[:k])
        hits = len(actual_set & predicted_set)
        precisions.append(hits / k)
        recalls.append(hits / len(actual_set) if len(actual_set) > 0 else 0)
    return np.mean(precisions), np.mean(recalls)

p_val_rf, r_val_rf = calculate_metrics(val_actual, rf_preds_val, k=20)

print(f"Precision@20 = {p_val_rf:.5f}")
print(f"Recall@20    = {r_val_rf:.5f}")

Precision@20 = 0.00390
Recall@20    = 0.02775


In [8]:
hgb_model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.08,
    max_depth=10,
    min_samples_leaf=10,
    early_stopping=True,
    random_state=RANDOM_STATE,
    scoring='roc_auc'
)

hgb_model.fit(X_train, y_train)

val_df['score_hgb'] = hgb_model.predict_proba(X_val)[:, 1]

print("🏆 Отбор персональных Топ-20...")
top_20_val_hgb = val_df.sort_values(['customer_id', 'score_hgb'], ascending=[True, False])
top_20_val_hgb = top_20_val_hgb.groupby('customer_id').head(20)

hgb_preds_val = top_20_val_hgb.groupby('customer_id')['article_id'].apply(list).to_dict()

p_val_hgb, r_val_hgb = calculate_metrics(val_actual, hgb_preds_val, k=20)

print(f"Precision@20 = {p_val_hgb:.5f}")
print(f"Recall@20    = {r_val_hgb:.5f}")

🏆 Отбор персональных Топ-20...
Precision@20 = 0.00341
Recall@20    = 0.02440


In [9]:
model_path_hgb = ARTIFACTS_DIR / "hgb_model.joblib"
joblib.dump(hgb_model, model_path_hgb)

meta_hgb = {
    "model_name": "HistGradientBoosting_Pointwise",
    "params": {
        "max_iter": 150,
        "learning_rate": 0.08,
        "max_depth": 10
    },
    "metrics_val": {
        "precision_at_20": p_val_hgb,
        "recall_at_20": r_val_hgb
    },
    "random_state": RANDOM_STATE
}

with open(ARTIFACTS_DIR / "hgb_model_meta.json", "w") as f:
    json.dump(meta_hgb, f, indent=4)

In [10]:
test_actual = test_data.groupby('customer_id')['article_id'].apply(list).to_dict()
test_users = list(test_actual.keys())

test_df = pd.DataFrame(
    list(itertools.product(test_users, top_100_candidates)), 
    columns=['customer_id', 'article_id']
)

test_df = test_df.merge(customers[customer_cols], on='customer_id', how='left')
test_df = test_df.merge(articles[article_cols], on='article_id', how='left')

test_df['age'] = test_df['age'].fillna(train_clf['age'].median())

for col in cat_cols:
    test_df[col] = test_df[col].fillna('Unknown')
    known_classes = set(label_encoders[col].classes_)
    test_df[col] = test_df[col].apply(lambda x: x if x in known_classes else 'Unknown')
    test_df[col] = label_encoders[col].transform(test_df[col])

X_test = test_df[features]


In [11]:
test_df['score_rf'] = rf_model.predict_proba(X_test)[:, 1]

top_20_test_rf = test_df.sort_values(['customer_id', 'score_rf'], ascending=[True, False]).groupby('customer_id').head(20)
rf_preds_test = top_20_test_rf.groupby('customer_id')['article_id'].apply(list).to_dict()

p_test_rf, r_test_rf = calculate_metrics(test_actual, rf_preds_test, k=20)

print(f"   Precision@20 = {p_test_rf:.5f} | Recall@20 = {r_test_rf:.5f}")

   Precision@20 = 0.00327 | Recall@20 = 0.02323


In [12]:
test_df['score_hgb'] = hgb_model.predict_proba(X_test)[:, 1]

top_20_test_hgb = test_df.sort_values(['customer_id', 'score_hgb'], ascending=[True, False]).groupby('customer_id').head(20)
hgb_preds_test = top_20_test_hgb.groupby('customer_id')['article_id'].apply(list).to_dict()

p_test_hgb, r_test_hgb = calculate_metrics(test_actual, hgb_preds_test, k=20)

print(f"   Precision@20 = {p_test_hgb:.5f} | Recall@20 = {r_test_hgb:.5f}")

   Precision@20 = 0.00283 | Recall@20 = 0.02005


In [13]:
ml_metrics = {
    "random_forest": {
        "val_precision_20": float(p_val_rf),
        "val_recall_20": float(r_val_rf),
        "test_precision_20": float(p_test_rf),
        "test_recall_20": float(r_test_rf)
    },
    "hist_gradient_boosting": {
        "val_precision_20": float(p_val_hgb),
        "val_recall_20": float(r_val_hgb),
        "test_precision_20": float(p_test_hgb),
        "test_recall_20": float(r_test_hgb)
    }
}

metrics_path = ARTIFACTS_DIR / "ml_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(ml_metrics, f, indent=4)

In [17]:
def objective_hgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_iter': trial.suggest_int('max_iter', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 20),
    }
    
    model = HistGradientBoostingClassifier(
        early_stopping=True,
        random_state=RANDOM_STATE,
        scoring='roc_auc',
        **params
    )
    
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    return score

study_hgb = optuna.create_study(direction='maximize')
study_hgb.optimize(objective_hgb, n_trials=10)

print(study_hgb.best_params)

[I 2026-05-30 19:41:15,157] A new study created in memory with name: no-name-3f0e52b2-d56a-4f24-b136-2e875bfacb35
[I 2026-05-30 19:41:42,370] Trial 0 finished with value: 0.7527583714634364 and parameters: {'learning_rate': 0.07420896874275619, 'max_iter': 155, 'max_depth': 12, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7527583714634364.
[I 2026-05-30 19:42:03,281] Trial 1 finished with value: 0.7525894682035316 and parameters: {'learning_rate': 0.10035218522380046, 'max_iter': 116, 'max_depth': 11, 'min_samples_leaf': 19}. Best is trial 0 with value: 0.7527583714634364.
[I 2026-05-30 19:42:23,393] Trial 2 finished with value: 0.7521806718795551 and parameters: {'learning_rate': 0.09914995580009138, 'max_iter': 109, 'max_depth': 14, 'min_samples_leaf': 11}. Best is trial 0 with value: 0.7527583714634364.
[I 2026-05-30 19:42:55,768] Trial 3 finished with value: 0.7492038041296064 and parameters: {'learning_rate': 0.042328929635924176, 'max_iter': 168, 'max_depth': 8, 'min_sam

{'learning_rate': 0.18322759052484025, 'max_iter': 141, 'max_depth': 12, 'min_samples_leaf': 11}


In [18]:
best_hgb_params = study_hgb.best_params

tuned_hgb_model = HistGradientBoostingClassifier(
    early_stopping=True,
    random_state=RANDOM_STATE,
    **best_hgb_params
)

tuned_hgb_model.fit(X_train, y_train)

val_df['score_tuned_hgb'] = tuned_hgb_model.predict_proba(X_val)[:, 1]

top_20_val_tuned = val_df.sort_values(['customer_id', 'score_tuned_hgb'], ascending=[True, False]).groupby('customer_id').head(20)
tuned_preds_val = top_20_val_tuned.groupby('customer_id')['article_id'].apply(list).to_dict()

p_val_tuned, r_val_tuned = calculate_metrics(val_actual, tuned_preds_val, k=20)

print(p_val_tuned)
print(r_val_tuned)

0.003481059336238686
0.024978631434855868


In [19]:
test_df['score_tuned_hgb'] = tuned_hgb_model.predict_proba(X_test)[:, 1]

top_20_test_tuned = test_df.sort_values(['customer_id', 'score_tuned_hgb'], ascending=[True, False]).groupby('customer_id').head(20)
tuned_preds_test = top_20_test_tuned.groupby('customer_id')['article_id'].apply(list).to_dict()

p_test_tuned, r_test_tuned = calculate_metrics(test_actual, tuned_preds_test, k=20)
print(p_test_tuned)
print(r_test_tuned)

0.002921264954094408
0.020654612378954187


In [20]:
joblib.dump(tuned_hgb_model, ARTIFACTS_DIR / "tuned_hgb_model.joblib")

ml_metrics = {
    "random_forest": {
        "val_precision_20": float(p_val_rf),
        "val_recall_20": float(r_val_rf),
        "test_precision_20": float(p_test_rf),
        "test_recall_20": float(r_test_rf)
    },
    "hist_gradient_boosting": {
        "val_precision_20": float(p_val_hgb),
        "val_recall_20": float(r_val_hgb),
        "test_precision_20": float(p_test_hgb),
        "test_recall_20": float(r_test_hgb)
    },
    "tuned_hist_gradient_boosting": {
        "val_precision_20": float(p_val_tuned),
        "val_recall_20": float(r_val_tuned),
        "test_precision_20": float(p_test_tuned),
        "test_recall_20": float(r_test_tuned)
    }
}

with open(ARTIFACTS_DIR / "ml_metrics.json", "w") as f:
    json.dump(ml_metrics, f, indent=4)